In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Criando o banco de dados Silver
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [0]:
# Definindo a janela para manter apenas o registro mais recente
win_consumidor = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

df_consumidores = spark.table("bronze.tb_customers") \
    .withColumn("row_number", F.row_number().over(win_consumidor)) \
    .filter("row_number = 1") \
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado")
    )

df_consumidores.write.format("delta").mode("overwrite").saveAsTable("silver.dim_consumidores")

In [0]:
from pyspark.sql import functions as F
from itertools import chain

# 1. Criando o mapeamento de tradução para o status
status_mapping = {
    "delivered": "entregue", 
    "canceled": "cancelado", 
    "shipped": "enviado", 
    "processing": "processando", 
    "invoiced": "faturado", 
    "unavailable": "indisponível", 
    "created": "criado", 
    "approved": "aprovado"
}

# Transformando o dicionário em colunas que o Spark entende
mapping_expr = F.create_map([F.lit(x) for x in chain(*status_mapping.items())])

# 2. Aplicando as transformações
df_pedidos = spark.table("bronze.tb_orders") \
    .withColumn("status", mapping_expr[F.col("order_status")]) \
    .withColumn("tempo_entrega_dias", F.datediff(F.col("order_delivered_customer_date"), F.col("order_purchase_timestamp"))) \
    .withColumn("tempo_entrega_estimado_dias", F.datediff(F.col("order_estimated_delivery_date"), F.col("order_purchase_timestamp"))) \
    .withColumn("diferenca_entrega_dias", F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")) \
    .withColumn("entrega_no_prazo", 
                F.when(F.col("order_status") != "delivered", "Não Entregue")
                .when(F.col("order_delivered_customer_date") <= F.col("order_estimated_delivery_date"), "Sim")
                .otherwise("Não"))

# 3. Salvando na Silver
df_pedidos.write.format("delta").mode("overwrite").saveAsTable("silver.fat_pedidos")

print("Tabela silver.fat_pedidos criada com sucesso!")

Tabela silver.fat_pedidos criada com sucesso!


In [0]:
from pyspark.sql import functions as F
from itertools import chain

# PARTE 1: ITENS DOS PEDIDOS 
print("Processando itens dos pedidos...")
spark.table("bronze.tb_order_items").select(
    F.col("order_id").alias("id_pedido"),
    F.col("order_item_id").alias("id_item"),
    F.col("product_id").alias("id_produto"),
    F.col("seller_id").alias("id_vendedor"),
    F.col("price").cast("double").alias("preco_BRL"),
    F.col("freight_value").cast("double").alias("preco_frete")
).write.format("delta").mode("overwrite").saveAsTable("silver.fat_itens_pedidos")

# PARTE 2: PAGAMENTOS (Com Tradução Corrigida) 
print("Processando pagamentos...")

# 1. Dicionário de tradução para pagamentos
payment_mapping = {
    "credit_card": "Cartão de Crédito", 
    "boleto": "Boleto", 
    "voucher": "Voucher", 
    "debit_card": "Cartão de Débito", 
    "not_defined": "Não Definido"
}

# 2. Criando a expressão de mapeamento para o Spark
pay_mapping_expr = F.create_map([F.lit(x) for x in chain(*payment_mapping.items())])

# 3. Aplicando a tradução e salvando
df_pagamentos = spark.table("bronze.tb_order_payments") \
    .withColumn("tipo_pagamento", pay_mapping_expr[F.col("payment_type")]) \
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("sequencial_pagamento"),
        F.col("tipo_pagamento"),
        F.col("payment_installments").alias("parcelas_pagamento"),
        F.col("payment_value").cast("double").alias("valor_pagamento")
    )

df_pagamentos.write.format("delta").mode("overwrite").saveAsTable("silver.fat_pagamentos_pedidos")

print("Tabelas silver.fat_itens_pedidos e silver.fat_pagamentos_pedidos criadas com sucesso!")

Processando itens dos pedidos...
Processando pagamentos...
Tabelas silver.fat_itens_pedidos e silver.fat_pagamentos_pedidos criadas com sucesso!


In [0]:
from pyspark.sql import functions as F

df_reviews = spark.table("bronze.tb_order_reviews") \
    .withColumn("data_criacao_valida", F.try_to_timestamp(F.col("review_creation_date"))) \
    .filter("data_criacao_valida <= current_timestamp() OR data_criacao_valida IS NULL") \
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.col("review_score").cast("int").alias("nota_avaliacao"),
        F.coalesce(F.col("review_comment_title"), F.lit("Sem título")).alias("titulo_avaliacao"),
        F.coalesce(F.col("review_comment_message"), F.lit("Sem comentário")).alias("comentario_avaliacao"),
        F.col("data_criacao_valida").alias("data_criacao_avaliacao"),
        F.try_to_timestamp(F.col("review_answer_timestamp")).alias("data_resposta_avaliacao")
    )

df_reviews.write.format("delta").mode("overwrite").saveAsTable("silver.fat_avaliacoes_pedidos")
print("Tabela silver.fat_avaliacoes_pedidos criada.")

Tabela silver.fat_avaliacoes_pedidos criada.


In [0]:
from pyspark.sql.window import Window

# Janelas de Deduplicação
win_prod = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())
win_sell = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

# 1. Dim Produtos
df_produtos = spark.table("bronze.tb_products") \
    .withColumn("rn", F.row_number().over(win_prod)) \
    .filter("rn = 1") \
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_category_name").alias("categoria_produto"),
        F.col("product_weight_g").cast("double").alias("peso_produto_gramas"),
        F.col("product_length_cm").cast("double").alias("comprimento_centimetros"),
        F.col("product_height_cm").cast("double").alias("altura_centimetros"),
        F.col("product_width_cm").cast("double").alias("largura_centimetros"),
        F.col("product_photos_qty").cast("int").alias("quantidade_fotos")
    )
df_produtos.write.format("delta").mode("overwrite").saveAsTable("silver.dim_produtos")

# 2. Dim Vendedores (com Upper Case)
df_vendedores = spark.table("bronze.tb_sellers") \
    .withColumn("rn", F.row_number().over(win_sell)) \
    .filter("rn = 1") \
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado")
    )
df_vendedores.write.format("delta").mode("overwrite").saveAsTable("silver.dim_vendedores")

print("Tabelas de Dimensões (Produtos e Vendedores) criadas com deduplicação.")

Tabelas de Dimensões (Produtos e Vendedores) criadas com deduplicação.


In [0]:
# 1. Tradução de Categorias
spark.table("bronze.tb_product_category_name_translation").select(
    F.col("product_category_name").alias("nome_produto_en"),
    F.col("product_category_name_english").alias("nome_produto_pt")
).write.format("delta").mode("overwrite").saveAsTable("silver.dim_categoria_produtos_traducao")

# 2. Cotação Dólar com preenchimento de furos
df_dolar_base = spark.table("bronze.tb_cotacao_dolar") \
    .withColumn("data_cota", F.to_date("dataHoraCotacao")) \
    .select("data_cota", "cotacaoCompra")

# Criando calendário contínuo para evitar buracos (finais de semana)
min_max = df_dolar_base.select(F.min("data_cota").alias("min_d"), F.max("data_cota").alias("max_d")).collect()
df_cal = spark.range(0, (min_max[0]['max_d'] - min_max[0]['min_d']).days + 1) \
    .withColumn("data_referencia", F.date_add(F.lit(min_max[0]['min_d']), F.col("id").cast("int")))

win_last_value = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, 0)

df_dolar_final = df_cal.join(df_dolar_base, df_cal.data_referencia == df_dolar_base.data_cota, "left") \
    .withColumn("cotacao_venda", F.last("cotacaoCompra", ignorenulls=True).over(win_last_value)) \
    .select(F.col("data_referencia").alias("data_cotacao"), F.round("cotacao_venda", 4).alias("cotacao_venda"))

df_dolar_final.write.format("delta").mode("overwrite").saveAsTable("silver.dim_cotacao_dolar")
print("Dólar processado com tratamento de finais de semana.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Dólar processado com tratamento de finais de semana.


In [0]:
from pyspark.sql import functions as F

# 1. Carregando as tabelas necessárias
df_f_pedidos = spark.table("silver.fat_pedidos")
df_f_pagamentos = spark.table("silver.fat_pagamentos_pedidos").groupBy("id_pedido").agg(F.sum("valor_pagamento").alias("total_brl"))
df_dim_dolar = spark.table("silver.dim_cotacao_dolar")

# 2. Fazendo os Joins com as chaves corretas e calculando os valores
df_pedido_total = df_f_pedidos.join(df_f_pagamentos, df_f_pedidos.order_id == df_f_pagamentos.id_pedido, "inner") \
    .withColumn("data_pedido", F.to_date("order_purchase_timestamp")) \
    .join(df_dim_dolar, F.col("data_pedido") == df_dim_dolar.data_cotacao, "left") \
    .select(
        F.col("order_id").alias("id_pedido"),         # Traduzindo para o formato final
        F.col("customer_id").alias("id_consumidor"),  # Traduzindo para o formato final
        "status",
        F.round("total_brl", 2).alias("valor_total_pago_brl"),
        F.round(F.col("total_brl") / F.col("cotacao_venda"), 2).alias("valor_total_pago_usd"),
        "data_pedido"
    )

# 3. Salvando a tabela final
df_pedido_total.write.format("delta").mode("overwrite").saveAsTable("silver.fat_pedido_total")
print("Tabela silver.fat_pedido_total criada com sucesso!")

# 4. Otimização Física (Delta Optimize com ZORDER)
print("Iniciando otimização do Delta Lake (ZORDER)...")
spark.sql("OPTIMIZE silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

# Como na fat_pedidos a coluna ainda está em inglês, otimizamos usando 'order_id'
spark.sql("OPTIMIZE silver.fat_pedidos ZORDER BY (order_id)")

print("--- CAMADA SILVER 100% CONCLUÍDA ---")

Tabela silver.fat_pedido_total criada com sucesso!
Iniciando otimização do Delta Lake (ZORDER)...
--- CAMADA SILVER 100% CONCLUÍDA ---
